# 02 — Fine-tuning experiments (E1, A1, A2, E7, E9, pairs) + evaluation

**Session options:** Accelerator **GPU**. Use the same type (T4 or P100) for every commit, and by everyone. Internet is not needed.

**Inputs (+ Add Input):**
1. `jbhi-experiments`: the scripts. **Latest version**: in the Input panel, use ⋮ → update if Kaggle offers a newer one.
2. `downtream-hybrid-ckpts-ppg-abp-ecg` (`phantom8`): the memmaps, plus the thesis fold models `hybrid_ckpts` and `hybrid_results.csv`.
3. `pretrain-v6-weights` (`phantom8`): **the thesis encoder**. Every run uses this one.
4. Model `jbhi-pretrained-weights`: only needed for the `pairs` group (the pair encoders).
5. The output of **`setup_1`** (*Notebooks → Your Work*): `splits/windows.csv` and the baseline runs.
6. **From the second commit on:** this notebook's own latest output, and any teammate's notebook-02 output. Everything found under `runs/` in the inputs is merged, and finished folds are never re-trained.

**How to run**
* **First time only (interactive, ~10 min):** run cells 1–3 with `DRY_RUN = True` in cell 3. Then set `DRY_RUN = False`.
* **Real runs:** choose `RUN_GROUPS` in cell 4, then **Save Version → Save & Run All (Commit)**. Every job stops cleanly before Kaggle's 12-h limit. If a job didn't finish, attach this notebook's newest output and commit again with the same groups.

## 1 — Find inputs, restore earlier results

In [ ]:
import glob, os, shutil, time, json
T0 = time.time()
DEADLINE = T0 + 11 * 3600          # finetune.py will not start a fold that would end after this

def find(pattern):
    return sorted(glob.glob(pattern, recursive=True))

s = find('/kaggle/input/**/finetune.py')
assert s, 'attach jbhi-experiments'
# several copies can be attached (old dataset versions, old notebook outputs): use the newest scripts
def version(f):
    v = os.path.join(os.path.dirname(f), 'VERSION')
    return int(open(v).read().strip()) if os.path.exists(v) else 0
s = sorted(s, key=version, reverse=True)
print('scripts from:', os.path.dirname(s[0]), '| version', version(s[0]), '| other copies ignored:', len(s) - 1)
shutil.copytree(os.path.dirname(s[0]), '/kaggle/working/exp', dirs_exist_ok=True)
os.chdir('/kaggle/working/exp')
assert os.path.exists('predict_thesis.py'), 'jbhi-experiments is an old version: update the input to the latest version'
assert '--max_flat_frac' in open('evaluate.py').read(), 'jbhi-experiments is an old version (evaluate.py has no --max_flat_frac): update the input to the latest version'
assert '--mask_start' in open('finetune.py').read(), 'jbhi-experiments is an old version (finetune.py has no --mask_start): attach jbhi_experiments_8'

mm = find('/kaggle/input/**/train/offset_5m/n_samples.npy')
assert mm, 'attach the memmap dataset'
MEM = os.path.dirname(os.path.dirname(os.path.dirname(mm[0])))

spl = find('/kaggle/input/**/splits/windows.csv')
assert spl, 'attach the output of setup_1 (Notebooks -> Your Work -> setup_1)'
SPL = spl[0]

enc = [p for p in find('/kaggle/input/**/hybrid_v4_encoders.pth') if 'pretrain-v6-weights' in p]
assert enc, 'attach pretrain-v6-weights (the thesis encoder)'
ENC = enc[0]
PAIRS = {os.path.basename(p).split('_encoders')[0].replace('ablation_', 'pair_'): p
         for p in find('/kaggle/input/**/ablation_*_encoders.pth')}

# encoders from the pretraining-ablation notebook (03): pretrain_<variant>_encoders.pth
ABL = {}
for p in find('/kaggle/input/**/pretrain_*_encoders.pth'):
    ABL.setdefault(os.path.basename(p)[len('pretrain_'):-len('_encoders.pth')], p)
BOLUS = os.path.join(os.path.dirname(SPL), 'controls_with_bolus.csv')

ck = find('/kaggle/input/**/hybrid_ckpts/')
THESIS_CKPT = ck[0] if ck else None
rc = find('/kaggle/input/**/hybrid_results.csv')
THESIS_RES = rc[0] if rc else None

RUNS = '/kaggle/working/runs'
os.makedirs(RUNS, exist_ok=True)
restored = 0
for d in find('/kaggle/input/**/runs/*/'):
    name = os.path.basename(os.path.normpath(d))
    if name.startswith('_'):
        continue
    shutil.copytree(d, os.path.join(RUNS, name), dirs_exist_ok=True)
    restored += 1

print('memmaps      :', MEM)
print('splits       :', SPL)
print('encoder      :', ENC)
print('pair encoders:', PAIRS or 'none attached')
print('ablation encoders (notebook 03):', ABL or 'none attached')
print('thesis ckpts :', THESIS_CKPT, '|', THESIS_RES)
print(f'restored {restored} run folders:', sorted(os.listdir(RUNS)))
if THESIS_RES:
    import pandas as pd
    r = pd.read_csv(THESIS_RES)
    if 'elapsed_s' in r:
        print('\nthesis run time per horizon (5 folds), for planning:',
              ', '.join(f'{w}: {e/60:.0f} min' for w, e in zip(r.window, r.elapsed_s)))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 — Re-score the thesis models (no training, a few minutes, once)

In [ ]:
if THESIS_CKPT and THESIS_RES and not os.path.exists(f'{RUNS}/thesis_original/summary.csv'):
    !python predict_thesis.py --memmap_dir "{MEM}" --windows_csv "{SPL}" --ckpt_dir "{THESIS_CKPT}" --results_csv "{THESIS_RES}" --out_dir {RUNS}
else:
    print('skipped (already done or thesis checkpoints not attached)')

## 3 — Dry run (interactive, first time only)

In [ ]:
DRY_RUN = False      # True the first time (interactive), then back to False before committing
if DRY_RUN:
    base = f'--memmap_dir "{MEM}" --windows_csv "{SPL}" --windows offset_5m --max_folds 1 --epochs 1 --max_train_batches 5 --out_dir /kaggle/working/dryrun'
    !python finetune.py {base} --init pretrained --encoder_path "{ENC}" --run_name dry_ssl
    !python finetune.py {base} --init scratch --run_name dry_scratch
    !python finetune.py {base} --init pretrained --encoder_path "{ENC}" --modalities ppg --run_name dry_single
    shutil.rmtree('/kaggle/working/dryrun', ignore_errors=True)

## 4 — Jobs for THIS commit

| Group | Runs | Fold trainings | Use |
|---|---|---|---|
| `main` | `trimodal_ssl`, `scratch` (6 horizons each) | 60 | E1, A1 / E4 |
| `frozen` | `frozen` (5 and 15 min) | 10 (fast) | A2 / E5 |
| `label_eff` | `le_ssl_*`, `le_scratch_*` (5 min; 10/25/50/100 %; 3 seeds) | 120 | E7 / Fig. 4 |
| `label_eff_extra` | `le_*_s3`, `le_*_s4` at 10 % and 25 % (5 min) | 40 (small) | 5 seeds where labels are scarce |
| `modalities` | `single_ppg/abp/ecg`, `drop_ppg/abp/ecg` (tri-modal encoder, one branch removed) | 180 | E9, pair configurations |
| `modalities_short` | same six runs, 5 and 15 min only | 60 | E9 on a time budget |
| `replicate15` | `drop_ecg_s1/s2`, `single_ppg_s1/s2` (15 min, new seeds) | 20 | checks the PPG+ABP 15-min result |
| `pairs` | `pair_abp_ecg`, `pair_abp_ppg`, `pair_ppg_ecg` (your pair-pretrained encoders) | 90 | thesis-style pairs |
| `mask15` | 15-min re-runs with the first 375 samples masked: `trimodal_ssl`, `scratch`, `frozen`, the six modality runs and the three pairs (`*_m15`) | 60 | 15-min results without the PPG start transient |
| `ablations` | `abl_<variant>` (5 min) and `abl_<variant>_m15` (15 min, masked) for every encoder from notebook 03 | 40 | A3–A6 (+ re-pretrained `full`) |
| `ablations_lowlabel` | `abq_<variant>_f0.25_s0..2` (5 min, 25 % of training patients, 3 seeds) | 75 (small) | A3–A6 where labels are scarce |

Suggested order: `main` → `label_eff` → `frozen` → `modalities`. Teammates can take whole groups in parallel, each in their own copy of this notebook.

In [ ]:
RUN_GROUPS = ['main']        # <- change per commit

COMMON = f'--memmap_dir "{MEM}" --windows_csv "{SPL}" --out_dir {RUNS} --resume --deadline {DEADLINE}'
P = f'--init pretrained --encoder_path "{ENC}"'
M15 = '--mask_start 375'      # mask the first 375 samples (15 patches) of every channel at 15 min
GROUPS = {
    'main': {'trimodal_ssl': f'{P} --keep_ckpts', 'scratch': '--init scratch'},
    'frozen': {'frozen': f'{P} --freeze_encoder --windows offset_5m offset_15m'},
    'label_eff': {f'le_{v}_f{f}_s{s}': (P if v == 'ssl' else '--init scratch') +
                  f' --windows offset_5m --label_frac {f} --seed {s}'
                  for f in [0.1, 0.25, 0.5, 1.0] for s in [0, 1, 2] for v in ['ssl', 'scratch']},
    'label_eff_extra': {f'le_{v}_f{f}_s{sd}': (P if v == 'ssl' else '--init scratch') +
                        f' --windows offset_5m --label_frac {f} --seed {sd}'
                        for f in [0.1, 0.25] for sd in [3, 4] for v in ['ssl', 'scratch']},
    'modalities': {**{f'single_{m}': f'{P} --modalities {m}' for m in ['ppg', 'abp', 'ecg']},
                   **{f'drop_{x}': f'{P} --modalities {",".join(m for m in ["ppg", "abp", "ecg"] if m != x)}'
                      for x in ['ppg', 'abp', 'ecg']}},
    'modalities_short': {**{f'single_{m}': f'{P} --modalities {m} --windows offset_5m offset_15m'
                            for m in ['ppg', 'abp', 'ecg']},
                         **{f'drop_{x}': f'{P} --modalities {",".join(m for m in ["ppg", "abp", "ecg"] if m != x)} '
                                         f'--windows offset_5m offset_15m' for x in ['ppg', 'abp', 'ecg']}},
    'replicate15': {**{f'drop_ecg_s{s}': f'{P} --modalities ppg,abp --windows offset_15m --seed {s}' for s in [1, 2]},
                    **{f'single_ppg_s{s}': f'{P} --modalities ppg --windows offset_15m --seed {s}' for s in [1, 2]}},
    'pairs': {name: f'--init pretrained --encoder_path "{p}" --modalities {",".join(name.split("_")[1:3])}'
              for name, p in PAIRS.items()},
    # 15-min runs use --mask_start 375: positive 15-min PPG windows have a start-of-window transient (edge_check.py)
    'mask15': {**{f'{n}_m15': f'{f} --windows offset_15m {M15}' for n, f in [
                   ('trimodal_ssl', P), ('scratch', '--init scratch'), ('frozen', f'{P} --freeze_encoder')]},
               **{f'single_{m}_m15': f'{P} --modalities {m} --windows offset_15m {M15}' for m in ['ppg', 'abp', 'ecg']},
               **{f'drop_{x}_m15': f'{P} --modalities {",".join(m for m in ["ppg", "abp", "ecg"] if m != x)} '
                                   f'--windows offset_15m {M15}' for x in ['ppg', 'abp', 'ecg']},
               **{f'{name}_m15': f'--init pretrained --encoder_path "{p}" --modalities {",".join(name.split("_")[1:3])} '
                                 f'--windows offset_15m {M15}' for name, p in PAIRS.items()}},
    'ablations': {**{f'abl_{t}': f'--init pretrained --encoder_path "{p}" --windows offset_5m'
                     for t, p in ABL.items() if t.endswith('_e30') or t == 'full'},
                  **{f'abl_{t}_m15': f'--init pretrained --encoder_path "{p}" --windows offset_15m {M15}'
                     for t, p in ABL.items() if t.endswith('_e30') or t == 'full'}},
    'ablations_lowlabel': {f'abq_{t}_f0.25_s{s}': f'--init pretrained --encoder_path "{p}" --windows offset_5m '
                           f'--label_frac 0.25 --seed {s}' for t, p in ABL.items() for s in [0, 1, 2]},
}

def done(name):
    return os.path.exists(f'{RUNS}/{name}/summary.csv')

# ---- run the jobs, one per GPU in parallel (Kaggle's T4 x2 gives two GPUs) ----
import subprocess, threading, queue, torch
N_GPU = max(1, torch.cuda.device_count())
LOGS = f'{RUNS}/_logs'
os.makedirs(LOGS, exist_ok=True)
todo = [(n, f) for g in RUN_GROUPS for n, f in GROUPS[g].items() if not done(n)]
print(f'{N_GPU} GPU(s); {len(todo)} job(s) to run:', [n for n, _ in todo])
print('already finished:', [n for g in RUN_GROUPS for n in GROUPS[g] if done(n)])
jobs = queue.Queue()
for j in todo:
    jobs.put(j)

def worker(gpu):
    while True:
        try:
            name, flags = jobs.get_nowait()
        except queue.Empty:
            return
        if time.time() > DEADLINE:
            print(f'[gpu{gpu}] deadline: not starting {name}', flush=True)
            continue
        print(f'[gpu{gpu}] start {name}', flush=True)
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        with open(f'{LOGS}/{name}.log', 'a') as lf:
            rc = subprocess.run(f'python finetune.py {COMMON} {flags} --run_name {name}', shell=True,
                                env=env, stdout=lf, stderr=subprocess.STDOUT).returncode
        print(f'[gpu{gpu}] end {name} (exit code {rc}){"" if rc == 0 else "  <-- ERROR, see log below"}', flush=True)
        if rc != 0:
            print(''.join(open(f'{LOGS}/{name}.log').readlines()[-30:]), flush=True)

def progress():
    for f in sorted(glob.glob(f'{LOGS}/*.log')):
        lines = [l.strip() for l in open(f) if 'TEST [' in l or '[deadline]' in l]
        print(f'  {os.path.basename(f)[:-4]:24s} {len(lines)} horizon(s) done' + (f' | last: {lines[-1]}' if lines else ''))

threads = [threading.Thread(target=worker, args=(g,)) for g in range(N_GPU)]
for t in threads:
    t.start()
while any(t.is_alive() for t in threads):
    for t in threads:
        t.join(timeout=600)          # status every 10 min
    print(f'\n--- {time.strftime("%H:%M")} ({(time.time() - T0) / 3600:.1f} h) ---', flush=True)
    progress()
print('\nall jobs for this commit have ended')

## 5 — Results so far (all runs in `runs/`, including baselines and the thesis models)

Test windows in which any channel is more than 10 % flat (no usable signal; the paper's quality rule) are left out of scoring: `--max_flat_frac 0.1`.

In [ ]:
import pandas as pd
REF = 'trimodal_ssl' if os.path.exists(f'{RUNS}/trimodal_ssl/summary.csv') else 'thesis_original'
!python evaluate.py --runs_dir {RUNS} --reference {REF} --memmap_dir "{MEM}" --max_flat_frac 0.1 > /kaggle/working/evaluate_log.txt 2>&1; tail -3 /kaggle/working/evaluate_log.txt
# sensitivity analysis: test set without the controls that received a vasopressor bolus
if os.path.exists(BOLUS):
    !python evaluate.py --runs_dir {RUNS} --reference {REF} --memmap_dir "{MEM}" --max_flat_frac 0.1 --exclude_cases "{BOLUS}" --out_dir {RUNS}/_eval_nobolus > /kaggle/working/evaluate_nobolus_log.txt 2>&1; tail -2 /kaggle/working/evaluate_nobolus_log.txt
for name in ['_eval', '_eval_nobolus']:
    f = f'{RUNS}/{name}/pooled_vs_reference.csv'
    if os.path.exists(f):
        print(f'\n{name}: mean AUROC over horizons, paired patient bootstrap vs {REF}')
        print(pd.read_csv(f).round(3).to_string(index=False))
r = pd.read_csv(f'{RUNS}/_eval/results_long.csv')
r = r[~r.run.str.startswith(('le_', 'abq_'))]
show = [c for c in ['run', 'window', 'auroc', 'auroc_lo', 'auroc_hi', 'auprc', 'delong_p', 'recall', 'specificity',
                    'mae_map', 'mae_sbp', 'mae_dbp'] if c in r]
print(f'reference for DeLong: {REF}')
print(r[show].round(3).to_string(index=False))

# 15-min runs with the start of the window masked (group mask15): DeLong vs the masked tri-modal model
if os.path.exists(f'{RUNS}/trimodal_ssl_m15/summary.csv'):
    !python evaluate.py --runs_dir {RUNS} --reference trimodal_ssl_m15 --memmap_dir "{MEM}" --max_flat_frac 0.1 --out_dir {RUNS}/_eval_m15 > /kaggle/working/evaluate_m15_log.txt 2>&1; tail -1 /kaggle/working/evaluate_m15_log.txt
    m = pd.read_csv(f'{RUNS}/_eval_m15/results_long.csv')
    m = m[(m.window == 'offset_15m') & ~m.run.str.startswith(('le_', 'abq_'))].sort_values('auroc', ascending=False)
    print('\n15 min, masked (*_m15) and unmasked runs; DeLong vs trimodal_ssl_m15')
    print(m[[c for c in show if c in m]].round(3).to_string(index=False))

# PAPER TABLES: 15-min results taken from the masked *_m15 runs (unmasked neural 15-min results dropped)
!python evaluate.py --runs_dir {RUNS} --reference {REF} --memmap_dir "{MEM}" --max_flat_frac 0.1 --masked_15 --out_dir {RUNS}/_eval_paper > /kaggle/working/evaluate_paper_log.txt 2>&1; tail -1 /kaggle/working/evaluate_paper_log.txt
if os.path.exists(BOLUS):
    !python evaluate.py --runs_dir {RUNS} --reference {REF} --memmap_dir "{MEM}" --max_flat_frac 0.1 --masked_15 --exclude_cases "{BOLUS}" --out_dir {RUNS}/_eval_paper_nobolus > /kaggle/working/evaluate_paper_nobolus_log.txt 2>&1; tail -1 /kaggle/working/evaluate_paper_nobolus_log.txt
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 300); pd.set_option('display.max_columns', 20)
for name in ['_eval_paper', '_eval_paper_nobolus']:
    if os.path.exists(f'{RUNS}/{name}/table_auroc.csv'):
        print(f'\n===== {name}: test AUROC (15 min = masked runs)')
        t = pd.read_csv(f'{RUNS}/{name}/table_auroc.csv', index_col=0)
        print(t[~t.index.str.startswith(('le_', 'abq_'))].to_string())
        for f in ['pooled_vs_reference.csv', 'label_efficiency.csv']:
            if os.path.exists(f'{RUNS}/{name}/{f}'):
                print(f'\n{name}/{f}'); print(pd.read_csv(f'{RUNS}/{name}/{f}').round(3).to_string(index=False))

In [ ]:
# which jobs are finished?
status = []
for g, jobs in GROUPS.items():
    for name in jobs:
        n_done = len(glob.glob(f'{RUNS}/{name}/summary_*.json'))
        status.append(dict(group=g, run=name, horizons_done=n_done, finished=done(name)))
pd.DataFrame(status).groupby('group').agg(runs=('run', 'size'), finished=('finished', 'sum'))

## 6 — Save

**Save Version → Save & Run All (Commit).** The output `runs/` holds every result so far. Fold checkpoints are kept only for `trimodal_ssl`; the natural-prevalence experiment needs them later.

**Next commit:** attach this notebook's newest output (input 6), set `RUN_GROUPS`, and commit again. Download `runs/_eval/` whenever you want the current tables (`table_auroc.tex`, `table_auprc.tex`, `results_long.csv`, `fig4_label_efficiency_offset_5m.pdf`).

In [ ]:
os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/exp', ignore_errors=True)
!du -sh /kaggle/working/runs